# Notebook 1 of 2 — Ingest: Internet → ADLS → Bronze

**Your job:** get NYC taxi and weather data from the internet into your lake, keep an untouched copy, then store it as **Bronze Delta** and register it in Unity Catalog.

> **Learning rule:** AI tools are allowed here. Understanding is not optional. You will explain every transformation in your own words in `OBSERVATION_REPORT.md`.

### The naming contract (do not improvise)

| Layer | ADLS path (inside container `lakehouse`) | Unity Catalog table |
|---|---|---|
| Raw taxi files | `raw/taxi/yellow_tripdata_2025-08.parquet` (also `-09`, `-10`) | — |
| Raw weather file | `raw/weather/nyc_daily_2025-08-01_2025-10-31.json` | — |
| Bronze taxi | `bronze/taxi_trips/` | `<catalog>.bronze.taxi_trips` |
| Bronze weather | `bronze/weather_daily/` | `<catalog>.bronze.weather_daily` |

All tables are **external Delta tables** pointing at the same storage account and container.

### Setup
Fill in the two `<...>` values, then run the cell. Use the existing catalog you were given (`SHOW CATALOGS` lists it). You cannot create your own.

In [ ]:
# ---- Fill in the two values marked <...> -------------------------------------
STORAGE_ACCOUNT = "<your_storage_account>"   # your ADLS Gen2 account name
CATALOG         = "<your_catalog>"           # run SHOW CATALOGS and use the existing one given to you

# ---- Do NOT change these: the names are part of the contract ------------------
CONTAINER = "lakehouse"
BASE      = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"
MONTHS    = ["2025-08", "2025-09", "2025-10"]

TAXI_URL     = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{month}.parquet"
WEATHER_FILE = "nyc_daily_2025-08-01_2025-10-31.json"
WEATHER_URL  = ("https://archive-api.open-meteo.com/v1/archive?latitude=40.7128&longitude=-74.006"
                "&start_date=2025-08-01&end_date=2025-10-31"
                "&daily=temperature_2m_max,temperature_2m_min,precipitation_sum,snowfall_sum,wind_speed_10m_max"
                "&timezone=America%2FNew_York")

---
## Step 1 — Land the raw files in ADLS

**What:** download 3 monthly taxi Parquet files and 1 weather JSON, and put them **unchanged** in the `raw/` folders from the contract table.

**Why:** if your Bronze logic turns out wrong, you rebuild from your own copy instead of hitting the internet again. Raw is the evidence.

| Source | Where it comes from |
|---|---|
| Taxi (3 files, ~60–75 MB each) | `TAXI_URL` with each month from `MONTHS` |
| Weather (daily, NYC, Aug–Oct 2025) | `WEATHER_URL` (Open-Meteo, free, no key) |

**Hints**
- Spark cannot read an `https://` address. Python downloads it; Spark reads it afterwards.
- You are on serverless: download to the driver's local disk (`/tmp`) with `requests`, then move it into the lake with `dbutils.fs.cp("file:/tmp/...", "abfss://...")`.
- Stream the download in chunks (`stream=True`) instead of loading 70 MB into memory. Call `raise_for_status()` so a bad URL fails loudly.
- Delete the temp file afterwards. Use `dbutils.fs.ls` to prove the files arrived and note their sizes.

**Pause and observe →** Why store an untouched copy in `raw/` instead of writing Delta straight from the download? Answer in report §1.

In [ ]:
# Step 1: your code here

---
## Step 2 — Bronze taxi: one Delta table for all three months

**Rules**
1. Read each month's raw Parquet and write to `bronze/taxi_trips/` as **Delta**.
2. Keep the **20 source columns exactly as they arrive** (names, types, values). No renaming, no casting, no filtering. Bronze is the evidence.
3. Add **three metadata columns**: `source_file` (string, the file name), `source_month` (string, `yyyy-MM`), `ingested_at` (timestamp).
4. Partition the table by `source_month`.
5. **Re-runnable:** running the cell twice must give the same row count, not double.

**Hints**
- Loop over `MONTHS`, one month at a time.
- `F.lit(...)` for constants, `F.current_timestamp()` for the load time.
- For rule 5 look at Delta's `replaceWhere` write option (with `mode("overwrite")`): it replaces only the month you are writing.
- Finish with `printSchema()` and a count per `source_month`.

**Pause and observe →** Which column types surprise you (look at `passenger_count`, the two timestamps, `RatecodeID`)? Why would a taxi meter count passengers in a `long`? Note row counts per month in report §2.

In [ ]:
# Step 2: your code here

---
## Step 3 — Bronze weather: flatten the JSON

**Rules**
1. The JSON contains one `daily` object made of **parallel arrays** (`time`, `temperature_2m_max`, …). Turn it into **one row per date**.
2. Keep the API's column names and values as they are. Add `source_file` and `ingested_at`.
3. Write Delta to `bronze/weather_daily/` (overwrite is fine; this one is a single file). Expect **92 rows**.

**Hints**
- JSON that spans many lines needs the `multiLine` read option.
- `select("daily.*")` gives you the arrays as columns. `F.arrays_zip` + `F.explode` turn parallel arrays into rows.

**Pause and observe →** Why is "one object with arrays" an awkward shape for analytics, and what does one row per date make easy? Report §2.

In [ ]:
# Step 3: your code here

---
## Step 4 — Register the Bronze external tables

Create schema `bronze` in your catalog, then register `taxi_trips` and `weather_daily` as **external Delta tables** over the folders from Steps 2 and 3 (you already know how). Print `Type`, `Provider` and `Location` for each.

**Hint:** if `CREATE SCHEMA` complains about managed storage, tell your instructor. Do not work around it by writing data somewhere else.

**Pause and observe →** Paste the `DESCRIBE EXTENDED` evidence in report §4. If you ran `DROP TABLE` on an external table, what happens to the files in ADLS? Answer from understanding; **do not test it on your data.**

In [ ]:
# Step 4: your code here

---
## Step 5 — Validate, then prove it is re-runnable

Run the check cell below (it is given). Fix anything marked ❌. Then **re-run your Step 2 cell** and run the checks again. Record the counts before and after in report §3.

> **Git checkpoint:** export or commit this notebook now (`git commit -m "notebook 1: bronze ingestion"`).

In [ ]:
from pyspark.sql import functions as F

_results = []
def check(label, ok):
    _results.append(bool(ok))
    print(("✅" if ok else "❌"), label)

def table_info(name):
    return {r.col_name: r.data_type for r in spark.sql(f"DESCRIBE EXTENDED {name}").collect()}

def summary():
    print(f"\n{sum(_results)} of {len(_results)} checks passed")

# ---------------- RAW ----------------
raw_taxi = sorted(f.name for f in dbutils.fs.ls(f"{BASE}/raw/taxi"))
check("raw/taxi holds exactly the 3 monthly parquet files",
      raw_taxi == [f"yellow_tripdata_{m}.parquet" for m in MONTHS])
check("raw/weather holds the weather JSON",
      [f.name for f in dbutils.fs.ls(f"{BASE}/raw/weather")] == [WEATHER_FILE])

# ---------------- BRONZE TAXI ----------------
t = f"{CATALOG}.bronze.taxi_trips"
info = table_info(t)
check("bronze.taxi_trips is an EXTERNAL table", info.get("Type") == "EXTERNAL")
check("bronze.taxi_trips points at <container>/bronze/taxi_trips", info.get("Location", "").rstrip("/").endswith("/bronze/taxi_trips"))
check("bronze.taxi_trips is Delta", str(info.get("Provider", "")).lower() == "delta")

bt = spark.table(t)
src_cols = ["VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime", "passenger_count", "trip_distance",
            "RatecodeID", "store_and_fwd_flag", "PULocationID", "DOLocationID", "payment_type", "fare_amount",
            "extra", "mta_tax", "tip_amount", "tolls_amount", "improvement_surcharge", "total_amount",
            "congestion_surcharge", "Airport_fee", "cbd_congestion_fee"]
check("Bronze keeps the 20 source column names untouched, plus 3 metadata columns",
      set(bt.columns) == set(src_cols + ["source_file", "source_month", "ingested_at"]))

per_month = {r.source_month: r["count"] for r in bt.groupBy("source_month").count().collect()}
raw_counts = {m: spark.read.parquet(f"{BASE}/raw/taxi/yellow_tripdata_{m}.parquet").count() for m in MONTHS}
check("Bronze rows per month == rows in the raw file (no loss, no duplicates)", per_month == raw_counts)

# ---------------- BRONZE WEATHER ----------------
w = f"{CATALOG}.bronze.weather_daily"
winfo = table_info(w)
check("bronze.weather_daily is an EXTERNAL table at <container>/bronze/weather_daily",
      winfo.get("Type") == "EXTERNAL" and winfo.get("Location", "").rstrip("/").endswith("/bronze/weather_daily"))
bw = spark.table(w)
check("weather: 92 rows (Aug 1 - Oct 31) and 92 distinct dates", bw.count() == 92 and bw.select("time").distinct().count() == 92)
check("weather keeps the API column names",
      {"time", "temperature_2m_max", "temperature_2m_min", "precipitation_sum", "snowfall_sum",
       "wind_speed_10m_max", "source_file", "ingested_at"} == set(bw.columns))

summary()
print("Bronze rows per month:", dict(sorted(per_month.items())))